In [87]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer

In [88]:
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv("data/test.csv")

df_test["Transported"] = False
df = pd.concat([df_train, df_test])
df.sample(5)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
3706,8073_01,Earth,True,G/1303/P,TRAPPIST-1e,24.0,False,0.0,0.0,0.0,0.0,0.0,Eddiey Davens,False
877,0939_01,Earth,True,G/135/P,NaN,15.0,False,0.0,0.0,0.0,0.0,0.0,Eulah Peter,True
5612,5971_01,Mars,False,F/1230/P,TRAPPIST-1e,47.0,False,448.0,0.0,891.0,0.0,235.0,Jayrin Kinad,False
8426,8999_01,Mars,False,D/274/S,55 Cancri e,32.0,False,2555.0,0.0,456.0,2.0,0.0,Roswal Raste,False
3932,4197_03,Earth,False,G/689/S,TRAPPIST-1e,19.0,False,1620.0,0.0,4.0,0.0,82.0,Harrie Cooks,False


In [89]:
df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='str')

In [90]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Cabin         12671 non-null  str    
 4   Destination   12696 non-null  str    
 5   Age           12700 non-null  float64
 6   VIP           12674 non-null  object 
 7   RoomService   12707 non-null  float64
 8   FoodCourt     12681 non-null  float64
 9   ShoppingMall  12664 non-null  float64
 10  Spa           12686 non-null  float64
 11  VRDeck        12702 non-null  float64
 12  Name          12676 non-null  str    
 13  Transported   12970 non-null  bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.4+ MB


In [91]:
df.isnull().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Name            294
Transported       0
dtype: int64

In [92]:
df.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,12700.000000,12707.000000,12681.000000,12664.000000,12686.000000,12702.000000
mean,28.771969,222.897852,451.961675,174.906033,308.476904,306.789482
std,14.387261,647.596664,1584.370747,590.558690,1130.279641,1180.097223
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,49.000000,77.000000,29.000000,57.000000,42.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [93]:
df.sample(5)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
3974,8665_01,Earth,True,G/1405/P,TRAPPIST-1e,19.0,False,0.0,0.0,0.0,0.0,0.0,Rayne Leblanchez,False
3169,3418_01,Mars,True,F/646/S,TRAPPIST-1e,72.0,False,NaN,0.0,0.0,0.0,0.0,Berers Meake,True
4983,5316_01,Mars,True,F/1087/P,55 Cancri e,23.0,False,0.0,0.0,0.0,0.0,0.0,Cats Ses,True
241,0509_01,Earth,True,NaN,TRAPPIST-1e,29.0,False,0.0,0.0,0.0,0.0,0.0,Alfrey Ewins,False
8140,8696_01,Europa,True,C/326/S,PSO J318.5-22,35.0,False,0.0,0.0,0.0,0.0,0.0,Bahoton Alvesssidy,True


In [94]:
df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split("/", expand=True)

In [95]:
df.drop(columns="Cabin", inplace=True)
df.sample(5)

,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,Num,Side
577,0600_01,Europa,False,TRAPPIST-1e,24.0,False,0.0,3385.0,826.0,9.0,793.0,Muonon Ormler,True,C,23,S
5119,5467_01,Earth,False,TRAPPIST-1e,31.0,False,2703.0,21.0,0.0,0.0,1.0,Evenna Garden,False,F,1046,S
3503,3761_01,NaN,False,55 Cancri e,45.0,False,824.0,2196.0,385.0,19.0,1113.0,Nuscham Stingive,True,E,222,P
1205,1286_01,Mars,True,TRAPPIST-1e,10.0,False,0.0,0.0,0.0,0.0,0.0,Jackix Fache,True,F,248,S
3786,4042_02,Europa,True,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,Etair Stewder,True,A,46,S


In [96]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Destination   12696 non-null  str    
 4   Age           12700 non-null  float64
 5   VIP           12674 non-null  object 
 6   RoomService   12707 non-null  float64
 7   FoodCourt     12681 non-null  float64
 8   ShoppingMall  12664 non-null  float64
 9   Spa           12686 non-null  float64
 10  VRDeck        12702 non-null  float64
 11  Name          12676 non-null  str    
 12  Transported   12970 non-null  bool   
 13  Deck          12671 non-null  str    
 14  Num           12671 non-null  str    
 15  Side          12671 non-null  str    
dtypes: bool(1), float64(6), object(2), str(7)
memory usage: 1.6+ MB


In [97]:
imputer = KNNImputer(n_neighbors=5, weights="distance")
num_cols = df[["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa","VRDeck"]]
filled_data = imputer.fit_transform(num_cols)

In [98]:
filled_data = pd.DataFrame(filled_data, columns=num_cols.columns, index=num_cols.index)
filled_data

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,39.000000,0.0,0.0,0.0,0.0,0.0
1,24.000000,109.0,9.0,25.0,549.0,44.0
2,58.000000,43.0,3576.0,0.0,6715.0,49.0
3,33.000000,0.0,1283.0,371.0,3329.0,193.0
4,16.000000,303.0,70.0,151.0,565.0,2.0
...,...,...,...,...,...,...
4272,34.000000,0.0,0.0,0.0,0.0,0.0
4273,42.000000,0.0,847.0,17.0,10.0,144.0
4274,35.200000,0.0,0.0,0.0,0.0,0.0
4275,23.666667,0.0,2680.0,0.0,0.0,523.0


In [99]:
df[num_cols.columns] = filled_data

In [100]:
df.isna().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Destination     274
Age               0
VIP             296
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name            294
Transported       0
Deck            299
Num             299
Side            299
dtype: int64

In [101]:
df.sample(10)

,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,Num,Side
1394,1460_02,Earth,True,PSO J318.5-22,13.0,False,0.0,0.0,0.0,0.0,0.0,Eduana Williotters,True,G,230,P
1530,1624_02,Earth,False,55 Cancri e,46.0,False,612.0,0.0,62.0,0.0,0.0,Dorice Barbes,True,G,250,S
1291,1375_01,Earth,False,TRAPPIST-1e,2.0,False,0.0,0.0,0.0,0.0,0.0,Aleen Sykess,False,G,208,S
670,1361_01,Mars,True,TRAPPIST-1e,24.0,False,0.0,0.0,0.0,0.0,0.0,Pus Sche,False,F,264,S
1358,1430_01,Earth,False,TRAPPIST-1e,14.0,False,360.0,0.0,2116.0,1.0,8.0,Lina Popez,False,F,283,P
431,0892_01,Europa,True,TRAPPIST-1e,17.0,False,0.0,0.0,0.0,0.0,0.0,Tarfik Poilsive,False,C,36,S
6793,7176_02,Europa,False,TRAPPIST-1e,26.0,False,0.0,6050.0,0.0,662.0,920.0,Kochard Chhaft,False,C,232,P
3224,7067_02,Earth,True,TRAPPIST-1e,5.0,False,0.0,0.0,0.0,0.0,0.0,Lis Wriggins,False,G,1158,S
1456,3125_01,Mars,False,TRAPPIST-1e,32.0,False,3337.0,6.0,1.0,0.0,0.0,Miten Pate,False,D,99,P
2246,2404_04,NaN,True,TRAPPIST-1e,44.0,False,0.0,0.0,0.0,0.0,0.0,Dira Disheract,True,C,86,S


In [102]:
df["Name"] = df["Name"].fillna('U')

In [103]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Destination   12696 non-null  str    
 4   Age           12970 non-null  float64
 5   VIP           12674 non-null  object 
 6   RoomService   12970 non-null  float64
 7   FoodCourt     12970 non-null  float64
 8   ShoppingMall  12970 non-null  float64
 9   Spa           12970 non-null  float64
 10  VRDeck        12970 non-null  float64
 11  Name          12970 non-null  str    
 12  Transported   12970 non-null  bool   
 13  Deck          12671 non-null  str    
 14  Num           12671 non-null  str    
 15  Side          12671 non-null  str    
dtypes: bool(1), float64(6), object(2), str(7)
memory usage: 1.6+ MB


In [104]:
df['CryoSleep'].value_counts()

CryoSleep
False    8079
True     4581
Name: count, dtype: int64

In [105]:
df['Destination'].value_counts()

Destination
TRAPPIST-1e      8871
55 Cancri e      2641
PSO J318.5-22    1184
Name: count, dtype: int64

In [106]:
df['HomePlanet'].value_counts()

HomePlanet
Earth     6865
Europa    3133
Mars      2684
Name: count, dtype: int64

In [107]:
df["VIP"].value_counts()

VIP
False    12401
True       273
Name: count, dtype: int64

In [108]:
df["VIP"] = df["VIP"].fillna(False)

In [109]:
df.isna().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Destination     274
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name              0
Transported       0
Deck            299
Num             299
Side            299
dtype: int64